In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-3-1-nonlinear-trajectory

Analyze mouse gene-expression trajectories across tissues.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


1-end

In [ ]:
import os
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages
import math
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from statsmodels.nonparametric.smoothers_lowess import lowess
import gseapy as gp
import time
import textwrap
import logging


INPUT_GENE_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
DATA_DIR = input_path("aging/scage/data/1-tissue-data")
GLOBAL_GENE_NAMES_FILE = input_path('header.txt')
OUTPUT_PLOTS_DIR = output_path('2-8.3-shanda/1-feature/1-figure/0-3-result-4-nonlinear/1.8-gene-trajectory')



AGE_GROUP_TO_MONTHS = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}

DEFAULT_N_CLUSTERS = 4
ENRICH_DB = 'GO_Biological_Process_2023'
API_SLEEP_INTERVAL = 1.5


os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(OUTPUT_PLOTS_DIR, 'run.log'), encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


def normalize_gene_name_mouse(name: str) -> str:
    name = name.strip()
    if not name: return name
    return name[0].upper() + name[1:].lower()

def get_global_gene_names() -> list:
    if not os.path.exists(GLOBAL_GENE_NAMES_FILE):
        logger.error(f"找不到 header 文件: {GLOBAL_GENE_NAMES_FILE}")
        return []
    with open(GLOBAL_GENE_NAMES_FILE, 'r') as f:
        return [normalize_gene_name_mouse(l) for l in f if l.strip()]

def load_selected_genes(file_path: str) -> list:
    with open(file_path, 'r') as f:
        return [normalize_gene_name_mouse(l) for l in f if l.strip()]

def format_go_term(term: str, width: int = 40) -> str:
    clean_term = term.split(' (GO:')[0]
    if clean_term:
        clean_term = clean_term[0].upper() + clean_term[1:]
    return textwrap.fill(clean_term, width=width)

def run_enrichr_with_retry(gene_list: list, gene_sets: str, organism: str = 'mouse',
                           max_retries: int = 3, retry_wait: float = 5.0) -> str:
    for attempt in range(1, max_retries + 1):
        try:
            enr = gp.enrichr(gene_list=gene_list, gene_sets=gene_sets, organism=organism, outdir=None, no_plot=True)
            sig_res = enr.results[enr.results['P-value'] < 0.05]
            if sig_res.empty: return ""
            top_term = sig_res.sort_values('P-value').iloc[0]['Term']
            return format_go_term(top_term, width=32)
        except Exception as e:
            logger.warning(f"  Enrichr 第 {attempt} 次请求失败: {e}")
            if attempt < max_retries: time.sleep(retry_wait)
    return ""

def plot_cluster(ax, cluster_genes: pd.DataFrame, unique_months: list,
                 cluster_id: int, go_text: str, cols: int):
    """
    Plot one cluster trajectory using actual time spacing on the x-axis.
    """
    num_genes = len(cluster_genes)


    for _, row in cluster_genes.iterrows():
        ax.plot(unique_months, row.values, color='grey', alpha=0.15, linewidth=0.25, zorder=1)

    cluster_mean = cluster_genes.mean(axis=0)


    if num_genes > 1:
        cluster_sem = cluster_genes.sem(axis=0)
        ax.fill_between(
            unique_months, cluster_mean - cluster_sem, cluster_mean + cluster_sem,
            color='#3498DB', alpha=0.25, linewidth=0, zorder=2
        )


    if len(unique_months) <= 2:
        ax.plot(unique_months, cluster_mean.values, color='black', linewidth=1.2, zorder=3)
    else:
        frac = min(0.8, max(0.4, 2.0 / len(unique_months)))
        smooth = lowess(cluster_mean.values, unique_months, frac=frac)
        ax.plot(smooth[:, 0], smooth[:, 1], color='black', linewidth=1.2, zorder=3)


    title_str = f"Cluster {cluster_id + 1} (n={num_genes})"
    if go_text:
        title_str += f"\n{go_text}"

    ax.set_title(title_str, fontsize=7.5, fontweight='bold', pad=6, loc='center', linespacing=1.2)


    tick_locs = np.arange(5, 35, 5)
    tick_labels = [f"{m}m" for m in tick_locs]
    
    ax.set_xticks(tick_locs)
    ax.set_xticklabels(tick_labels, fontsize=7)
    

    ax.set_xlim(0, 32)
    # =========================================================================


    ax.axhline(0, color='black', linestyle='--', linewidth=0.4, alpha=0.3, zorder=0)
    ax.set_ylim(-2.5, 2.5)
    

    ax.grid(True, axis='x', color='grey', linestyle='-', linewidth=0.2, alpha=0.2, zorder=0)


    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if cluster_id % cols == 0:
        ax.set_ylabel("Scaled expression", fontsize=8, fontweight='bold')




BASE_SIZE = 7 
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.unicode_minus': False,
    'font.size': BASE_SIZE,
    'axes.labelsize': BASE_SIZE,
    'ytick.labelsize': BASE_SIZE,
    'xtick.labelsize': BASE_SIZE,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5
})

all_gene_names = get_global_gene_names()
if not all_gene_names:
    logger.error("无法加载 header 文件，程序终止。")
    exit()

gene_to_idx = {name: i for i, name in enumerate(all_gene_names)}
gene_files = [f for f in os.listdir(INPUT_GENE_DIR) if f.endswith('.txt')]

failed_tissues = []
logger.info(f"共发现 {len(gene_files)} 个组织文件，开始处理...")

for g_file in tqdm(gene_files, desc="Processing Tissues"):
    tissue_name = (g_file.split('_Knee_')[0] if '_Knee_' in g_file else g_file.split('_Final_')[0])
    
    try:
        target_genes = load_selected_genes(os.path.join(INPUT_GENE_DIR, g_file))
        valid_genes = [g for g in target_genes if g in gene_to_idx]
        if not valid_genes: continue

        hdf5_path = os.path.join(DATA_DIR, f"{tissue_name}.hdf5")
        if not os.path.exists(hdf5_path): hdf5_path = os.path.join(DATA_DIR, f"{tissue_name}.h5")
        if not os.path.exists(hdf5_path): continue

        with h5py.File(hdf5_path, 'r') as f:
            sorted_pairs = sorted([(gene_to_idx[g], g) for g in valid_genes])
            indices = [p[0] for p in sorted_pairs]
            plot_genes = [p[1] for p in sorted_pairs]
            raw_data = f['data'][:, indices]
            data_matrix = np.log1p(raw_data)
            labels = f['label'][:, 0]

        df = pd.DataFrame(data_matrix, columns=plot_genes)

        df['Age_Months'] = pd.Series(labels).map(AGE_GROUP_TO_MONTHS)
        df = df.dropna(subset=['Age_Months'])


        grouped_mean = df.groupby('Age_Months')[plot_genes].mean()
        unique_months = sorted(df['Age_Months'].unique())

        current_n_clusters = 2 if len(unique_months) <= 2 else DEFAULT_N_CLUSTERS
        traj_df = grouped_mean.T.dropna()

        if len(traj_df) < current_n_clusters: continue

        scaler = StandardScaler()
        z_scores = scaler.fit_transform(traj_df.T).T
        z_df = pd.DataFrame(z_scores, index=traj_df.index, columns=unique_months)

        hc = AgglomerativeClustering(n_clusters=current_n_clusters, linkage='ward')
        z_df['Cluster'] = hc.fit_predict(z_scores)

        cols = min(current_n_clusters, 4)
        rows = math.ceil(current_n_clusters / cols)


        if cols <= 2: fig_w = 85 / 25.4
        elif cols == 3: fig_w = 114 / 25.4
        else: fig_w = 170 / 25.4

        fig_h = (45 * rows) / 25.4 

        fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))

        if current_n_clusters == 1: axes = np.array([axes])
        else: axes = np.array(axes).flatten()

        for cluster_id in range(current_n_clusters):
            ax = axes[cluster_id]
            cluster_genes_df = z_df[z_df['Cluster'] == cluster_id].drop(columns='Cluster')
            gene_list = cluster_genes_df.index.tolist()
            num_genes = len(gene_list)

            go_text = ""
            if num_genes >= 5:
                go_text = run_enrichr_with_retry(gene_list=gene_list, gene_sets=ENRICH_DB, organism='mouse')


            plot_cluster(ax=ax, cluster_genes=cluster_genes_df, unique_months=unique_months,
                         cluster_id=cluster_id, go_text=go_text, cols=cols)

            time.sleep(API_SLEEP_INTERVAL)


        for j in range(current_n_clusters, len(axes)):
            fig.delaxes(axes[j])


        fig.tight_layout(pad=1.0)


        save_pdf_path = os.path.join(OUTPUT_PLOTS_DIR, f"{tissue_name}_Annotated_Clusters_TrueTime.pdf")
        with PdfPages(save_pdf_path) as pdf:
            pdf.savefig(fig, bbox_inches='tight', transparent=True)

        plt.close(fig)

        csv_path = os.path.join(OUTPUT_PLOTS_DIR, f"{tissue_name}_Cluster_Assignments.csv")
        z_df['Cluster'].to_csv(csv_path)

        logger.info(f"[{tissue_name}] ✅ 完成，基因数={len(valid_genes)}")

    except Exception as e:
        logger.error(f"[{tissue_name}] ❌ 处理失败: {e}", exc_info=True)
        failed_tissues.append((tissue_name, str(e)))
        plt.close('all') 

if failed_tissues:
    log_path = os.path.join(OUTPUT_PLOTS_DIR, 'failed_tissues.log')
    with open(log_path, 'w', encoding='utf-8') as f:
        f.write("tissue_name\terror_message\n")
        for t, e in failed_tissues:
            f.write(f"{t}\t{e}\n")
    logger.warning(f"以下 {len(failed_tissues)} 个组织处理失败，详见: {log_path}\n")
else:
    logger.info("所有组织均处理成功，无失败记录。")

logger.info(f"✅ 全部完成！符合主刊定标、并以真实月份为 X 轴的图表已保存至: {OUTPUT_PLOTS_DIR}")

Draft heatmap section; author review remains necessary.

In [ ]:
import os
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import gseapy as gp
import time
import textwrap
import logging


INPUT_GENE_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
DATA_DIR = input_path("aging/scage/data/1-tissue-data")
GLOBAL_GENE_NAMES_FILE = input_path('header.txt')
OUTPUT_PLOTS_DIR = output_path('2-8.3-shanda/1-feature/1-figure/0-3-result-4-nonlinear/1-multi-tissue-gene-trajectory')

AGE_GROUP_TO_MONTHS = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}
UNIQUE_MONTHS = [1, 3, 18, 21, 24, 30]

CLUSTERS_PER_TISSUE = 4
ENRICH_DB = 'GO_Biological_Process_2023'
API_SLEEP_INTERVAL = 1.0 


TISSUE_COLORS = {
    'Bladder': '#1f77b4', 'Brain_Myeloid': '#aec7e8', 'Brain_Non-Myeloid': '#ff7f0e',
    'Fat': '#ffbb78', 'Heart_and_Aorta': '#2ca02c', 'Heart': '#98df8a',
    'Kidney': '#d62728', 'Large_Intestine': '#ff9896', 'Limb_Muscle': '#9467bd',
    'Liver': '#c5b0d5', 'Lung': '#8c564b', 'Mammary_Gland': '#c49c94',
    'Marrow': '#e377c2', 'Pancreas': '#f7b6d2', 'Skin': '#7f7f7f',
    'Spleen': '#bcbd22', 'Thymus': '#dbdb8d', 'Tongue': '#17becf', 'Trachea': '#9edae5'
}

os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

BASE_SIZE = 7 
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica'],
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.unicode_minus': False,
    'font.size': BASE_SIZE
})


def normalize_gene_name_mouse(name: str) -> str:
    name = name.strip()
    return name[0].upper() + name[1:].lower() if name else name

def format_go_term(term: str, width: int = 40) -> str:
    clean_term = term.split(' (GO:')[0]
    return textwrap.fill(clean_term[0].upper() + clean_term[1:] if clean_term else "", width=width)

def run_enrichr_with_retry(gene_list: list, max_retries: int = 3) -> str:
    if len(gene_list) < 5: return ""
    for attempt in range(1, max_retries + 1):
        try:
            enr = gp.enrichr(gene_list=gene_list, gene_sets=ENRICH_DB, organism='mouse', outdir=None, no_plot=True)
            sig_res = enr.results[enr.results['P-value'] < 0.05]
            if not sig_res.empty:
                return format_go_term(sig_res.sort_values('P-value').iloc[0]['Term'], width=45)
            return ""
        except Exception:
            if attempt < max_retries: time.sleep(2.0)
    return ""


def main():

    with open(GLOBAL_GENE_NAMES_FILE, 'r') as f:
        gene_to_idx = {normalize_gene_name_mouse(l): i for i, l in enumerate(f) if l.strip()}
    
    gene_files = [f for f in os.listdir(INPUT_GENE_DIR) if f.endswith('.txt')]
    
    meta_clusters_data = []
    
    logger.info("第一阶段：遍历 19 个组织，提取每个 Cluster 的中心轨迹与通路注释...")
    for g_file in tqdm(gene_files, desc="Extracting Features"):
        tissue_name = (g_file.split('_Knee_')[0] if '_Knee_' in g_file else g_file.split('_Final_')[0])
        with open(os.path.join(INPUT_GENE_DIR, g_file), 'r') as f:
            valid_genes = [g for g in (normalize_gene_name_mouse(l) for l in f if l.strip()) if g in gene_to_idx]
        if not valid_genes: continue

        hdf5_path = os.path.join(DATA_DIR, f"{tissue_name}.hdf5")
        if not os.path.exists(hdf5_path): hdf5_path = os.path.join(DATA_DIR, f"{tissue_name}.h5")
        if not os.path.exists(hdf5_path): continue

        with h5py.File(hdf5_path, 'r') as f:
            indices = [gene_to_idx[g] for g in valid_genes]
            raw_data = f['data'][:, indices]
            labels = f['label'][:, 0]

        df = pd.DataFrame(np.log1p(raw_data), columns=valid_genes)
        label_to_month = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}
        df['Age_Months'] = pd.Series(labels).map(label_to_month)
        

        grouped_mean = df.groupby('Age_Months').mean().reindex(UNIQUE_MONTHS).interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
        scaler = StandardScaler()
        z_scores = scaler.fit_transform(grouped_mean)
        tissue_z_df = pd.DataFrame(z_scores, index=grouped_mean.index, columns=grouped_mean.columns)
        

        current_n_clusters = min(CLUSTERS_PER_TISSUE, len(valid_genes) // 5)
        if current_n_clusters < 2: current_n_clusters = 2
            
        hc = AgglomerativeClustering(n_clusters=current_n_clusters, linkage='ward')
        cluster_labels = hc.fit_predict(tissue_z_df.T)
        
        for c_id in range(current_n_clusters):
            c_genes = tissue_z_df.columns[cluster_labels == c_id].tolist()
            if not c_genes: continue
            

            mean_traj = tissue_z_df[c_genes].mean(axis=1).values
            

            go_text = run_enrichr_with_retry(c_genes)
            time.sleep(API_SLEEP_INTERVAL)
            

            meta_id = f"{tissue_name}_C{c_id+1}"
            meta_info = {
                'ID': meta_id,
                'Tissue': tissue_name,
                'Gene_Count': len(c_genes),
                'GO_Term': go_text
            }

            for idx, m in enumerate(UNIQUE_MONTHS):
                meta_info[f"{m}m"] = mean_traj[idx]
                
            meta_clusters_data.append(meta_info)


    meta_df = pd.DataFrame(meta_clusters_data).set_index('ID')
    meta_df.to_csv(os.path.join(OUTPUT_PLOTS_DIR, "Meta_Clusters_Data.csv"))
    

    time_cols = [f"{m}m" for m in UNIQUE_MONTHS]
    data_matrix = meta_df[time_cols].astype(float)
    

    logger.info("第二阶段：执行全局 Meta-clustering 并绘制主刊级复杂热图...")
    


    tissue_color_map = meta_df['Tissue'].map(TISSUE_COLORS)
    tissue_color_map.name = 'Tissue'
    

    norm_counts = (meta_df['Gene_Count'] - meta_df['Gene_Count'].min()) / (meta_df['Gene_Count'].max() - meta_df['Gene_Count'].min())
    count_cmap = mpl.cm.get_cmap('Purples')
    count_color_map = norm_counts.apply(count_cmap)
    count_color_map.name = 'Gene Size'
    
    row_colors = pd.concat([tissue_color_map, count_color_map], axis=1)


    fig_w, fig_h = 200 / 25.4, 250 / 25.4
    

    g = sns.clustermap(
        data_matrix,
        cmap='RdYlBu_r',
        center=0,
        row_cluster=True,
        col_cluster=False,
        row_colors=row_colors,
        figsize=(fig_w, fig_h),
        dendrogram_ratio=(0.15, 0.0),
        cbar_pos=(0.02, 0.85, 0.03, 0.1),
        linewidths=0.2, linecolor='white'
    )
    


    g.ax_heatmap.set_ylabel('')
    g.ax_heatmap.set_xlabel('Age (Months)', fontweight='bold')
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=0)
    g.ax_heatmap.set_yticklabels([])
    g.ax_heatmap.tick_params(axis='y', which='both', left=False, right=False)


    g.ax_cbar.set_title('Mean Z-score', fontsize=7, fontweight='bold', pad=10)



    row_order = g.dendrogram_row.reordered_ind
    ordered_ids = data_matrix.index[row_order]
    
    for i, idx in enumerate(ordered_ids):
        row_data = meta_df.loc[idx]
        tissue_name = row_data['Tissue']
        go_text = row_data['GO_Term']
        

        if go_text:

            clean_go_text = go_text.replace('\n', ' ')
            anno_text = f"[{tissue_name}] {clean_go_text}"
        else:
            anno_text = f"[{tissue_name}] Unknown/Unenriched"

        g.ax_heatmap.text(
            data_matrix.shape[1] + 0.1,
            i + 0.5,
            anno_text,
            va='center',
            ha='left',
            fontsize=5.5,
            color='#333333'
        )


    legend_elements = [plt.Rectangle((0,0),1,1, color=color, label=tissue) 
                       for tissue, color in TISSUE_COLORS.items()]

    g.ax_row_dendrogram.legend(
        handles=legend_elements, title="Tissues",
        loc="lower left", bbox_to_anchor=(-1.0, 0.0),
        ncol=1, fontsize=5.5, title_fontsize=6.5, frameon=False
    )


    out_pdf = os.path.join(OUTPUT_PLOTS_DIR, "Figure_2C_Meta_Module_Heatmap.pdf")

    g.savefig(out_pdf, bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    
    logger.info(f"✅ 全局 Meta-Module 复杂热图绘制完成！已保存至: {out_pdf}")
    logger.info("提示：请查看右侧的 GO Term 注释，聚在一起的红色或蓝色大区块，代表了跨组织的共性衰老通路！")

if __name__ == "__main__":
    main()

Global view of expression trajectories for all selected genes.

In [ ]:
import os
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.interpolate import PchipInterpolator
from scipy.stats import pearsonr
from statsmodels.nonparametric.smoothers_lowess import lowess 
import gseapy as gp
import time
import textwrap
import logging
import math
from tqdm import tqdm


INPUT_GENE_DIR = input_path("2-8.3-shanda/1-feature/9-Final_Segmented_Genes")
DATA_DIR = input_path("aging/scage/data/1-tissue-data")
GLOBAL_GENE_NAMES_FILE = input_path('header.txt')
OUTPUT_PLOTS_DIR = output_path('2-8.3-shanda/1-feature/1-figure/0-3-result-4-nonlinear/1-multi-gene-trajectory')

AGE_GROUP_TO_MONTHS = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}
UNIQUE_MONTHS = sorted(list(AGE_GROUP_TO_MONTHS.values()))


K_MIN = 8
K_MAX = 16
ENRICH_DB = 'GO_Biological_Process_2023'


MIN_EXPR_THRESHOLD = 0.05 

MIN_TIMEPOINTS_REQUIRED = 3

TISSUE_COLORS = {
    'Bladder': '#1f77b4', 'Brain_Myeloid': '#aec7e8', 'Brain_Non-Myeloid': '#ff7f0e',
    'Fat': '#ffbb78', 'Heart_and_Aorta': '#2ca02c', 'Heart': '#98df8a',
    'Kidney': '#d62728', 'Large_Intestine': '#ff9896', 'Limb_Muscle': '#9467bd',
    'Liver': '#c5b0d5', 'Lung': '#8c564b', 'Mammary_Gland': '#c49c94',
    'Marrow': '#e377c2', 'Pancreas': '#f7b6d2', 'Skin': '#7f7f7f',
    'Spleen': '#bcbd22', 'Thymus': '#dbdb8d', 'Tongue': '#17becf', 'Trachea': '#9edae5'
}

os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica'],
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.unicode_minus': False,
    'font.size': 7, 'axes.labelsize': 7, 'ytick.labelsize': 7, 'xtick.labelsize': 7,
    'legend.fontsize': 6.5, 'axes.linewidth': 0.5,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2.0, 'ytick.major.size': 2.0
})

def smooth_trajectory(x, y, x_dense, frac=0.6):
    try:
        y_smoothed_points = lowess(y, x, frac=frac, return_sorted=False)
        pchip = PchipInterpolator(x, y_smoothed_points)
        return pchip(x_dense)
    except Exception:
        return np.interp(x_dense, x, y)

def format_go_term_wrapped(term: str, width: int = 25) -> str:
    clean_term = term.split(' (GO:')[0]
    if not clean_term: return ""
    clean_term = clean_term[0].upper() + clean_term[1:]
    return textwrap.fill(clean_term, width=width)

def run_enrichr_with_retry(gene_list: list) -> str:
    if len(gene_list) < 5: return ""
    for attempt in range(1, 4):
        try:
            enr = gp.enrichr(gene_list=gene_list, gene_sets=ENRICH_DB, organism='mouse', outdir=None, no_plot=True)
            sig_res = enr.results[enr.results['P-value'] < 0.05]
            if not sig_res.empty: 
                result = format_go_term_wrapped(sig_res.sort_values('P-value').iloc[0]['Term'])
                time.sleep(1.0)
                return result
            time.sleep(1.0)
            return ""
        except Exception: 
            time.sleep(1.5)
    return ""


def main():
    with open(GLOBAL_GENE_NAMES_FILE, 'r') as f:
        gene_to_idx = {l.strip()[0].upper() + l.strip()[1:].lower() if l.strip() else l: i for i, l in enumerate(f) if l.strip()}
    
    gene_files = [f for f in os.listdir(INPUT_GENE_DIR) if f.endswith('.txt')]
    multi_tissue_z_data = {}
    actual_tissues_found = []
    
    logger.info("Step 1: 执行底噪过滤与数据完整性校验，构建安全 Z-score 矩阵...")
    for g_file in tqdm(gene_files):
        tissue_name = (g_file.split('_Knee_')[0] if '_Knee_' in g_file else g_file.split('_Final_')[0])
        with open(os.path.join(INPUT_GENE_DIR, g_file), 'r') as f:
            valid_genes = [g for g in (l.strip()[0].upper() + l.strip()[1:].lower() for l in f if l.strip()) if g in gene_to_idx]
        if not valid_genes: continue

        hdf5_path = os.path.join(DATA_DIR, f"{tissue_name}.hdf5")
        if not os.path.exists(hdf5_path): hdf5_path = os.path.join(DATA_DIR, f"{tissue_name}.h5")
        if not os.path.exists(hdf5_path): continue

        with h5py.File(hdf5_path, 'r') as f:
            indices = [gene_to_idx[g] for g in valid_genes]
            raw_data = f['data'][:, indices]
            labels = f['label'][:, 0]

        df_raw = pd.DataFrame(raw_data, columns=valid_genes)
        label_to_month = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}
        df_raw['Age_Months'] = pd.Series(labels).map(label_to_month)
        df_raw = df_raw.dropna(subset=['Age_Months'])
        
        if df_raw['Age_Months'].nunique() < MIN_TIMEPOINTS_REQUIRED:
            continue
            
        gene_means = df_raw.drop(columns=['Age_Months']).mean()
        high_expr_genes = gene_means[gene_means > MIN_EXPR_THRESHOLD].index.tolist()
        if not high_expr_genes: continue
        
        if tissue_name not in actual_tissues_found:
            actual_tissues_found.append(tissue_name)

        df_log = np.log1p(df_raw[high_expr_genes].astype(float))
        df_log['Age_Months'] = df_raw['Age_Months'].values
        
        grouped_mean = df_log.groupby('Age_Months').mean().reindex(UNIQUE_MONTHS)
        grouped_mean = grouped_mean.interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
        
        z_scores = StandardScaler().fit_transform(grouped_mean)
        multi_tissue_z_data[tissue_name] = pd.DataFrame(z_scores, index=grouped_mean.index, columns=grouped_mean.columns)

    logger.info("Step 2: 构建无噪音全局共识表达矩阵...")
    all_genes_union = list(set().union(*(df.columns for df in multi_tissue_z_data.values())))
    global_z_matrix = pd.DataFrame(0.0, index=UNIQUE_MONTHS, columns=all_genes_union)
    gene_counts = pd.Series(0, index=all_genes_union)
    
    for z_df in multi_tissue_z_data.values():
        global_z_matrix[z_df.columns] = global_z_matrix[z_df.columns].add(z_df, fill_value=0)
        gene_counts[z_df.columns] += 1
        
    global_z_matrix = global_z_matrix.div(gene_counts, axis=1).dropna(axis=1)
    
    logger.info(f"Step 3: 自动评估最佳聚类数 K (范围: {K_MIN} 到 {K_MAX})...")
    X = global_z_matrix.T.values
    best_k = K_MIN
    best_sil_score = -1
    n_samples_for_eval = min(2000, X.shape[0])
    
    for k in tqdm(range(K_MIN, K_MAX + 1), desc="Evaluating optimal K"):
        hc_temp = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels_temp = hc_temp.fit_predict(X)
        
        sil_scores_sub = []
        for seed in [42, 123, 2024]:
            s = silhouette_score(X, labels_temp, sample_size=n_samples_for_eval, random_state=seed)
            sil_scores_sub.append(s)
        sil_score = np.mean(sil_scores_sub)
        
        if sil_score > best_sil_score:
            best_sil_score = sil_score
            best_k = k
            
    logger.info(f"🏆 算法判定最优模块数: K = {best_k} (轮廓系数: {best_sil_score:.4f})")
    
    hc_final = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
    final_labels = hc_final.fit_predict(X)
    modules_dict = {f"Module {i+1}": global_z_matrix.columns[final_labels == i].tolist() for i in range(best_k)}

    module_deltas = {}
    for key, mod_genes in modules_dict.items():
        traj_mean = global_z_matrix[mod_genes].mean(axis=1)
        module_deltas[key] = traj_mean.iloc[-1] - traj_mean.iloc[0]
    
    sorted_module_keys = sorted(modules_dict.keys(), key=lambda k: module_deltas[k])

    logger.info(f"Step 4: 根据演化规律生成渐变式排版矩阵与防碰撞标注...")
    cols = 4
    rows = math.ceil(best_k / cols) 
    fig_w = 178 / 25.4
    fig_h = (rows * 45 + 60) / 25.4 
    

    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), sharex=False, sharey=False)
    
    for i in range(best_k, rows * cols):
        fig.delaxes(axes.flatten()[i])
        
    x_smooth = np.linspace(min(UNIQUE_MONTHS), max(UNIQUE_MONTHS), 300) 
    
    pe_white_border = [path_effects.Stroke(linewidth=2.5, foreground='white'), path_effects.Normal()]
    pe_white_border_mean = [path_effects.Stroke(linewidth=1.8, foreground='white'), path_effects.Normal()]
    
    for idx, key in enumerate(tqdm(sorted_module_keys, desc="Plotting Ordered Modules")):
        ax = axes.flatten()[idx]
        mod_genes = modules_dict[key]
        go_text = run_enrichr_with_retry(mod_genes)
        
        ax.set_box_aspect(1)
        title_color = '#333333'
        
        tissue_trajs = {}
        for tissue in TISSUE_COLORS.keys():
            tissue_df = multi_tissue_z_data.get(tissue)
            if tissue_df is None: continue 
            
            valid_genes = [g for g in mod_genes if g in tissue_df.columns]
            if not valid_genes: continue
            tissue_trajs[tissue] = tissue_df[valid_genes].mean(axis=1).values
            
        if not tissue_trajs: continue
        
        all_trajs_matrix = np.array(list(tissue_trajs.values()))
        consensus_traj = np.mean(all_trajs_matrix, axis=0)
        consensus_std = np.std(all_trajs_matrix, axis=0) 
        
        correlations = {}
        for t, traj in tissue_trajs.items():
            other_trajs = [v for k, v in tissue_trajs.items() if k != t]
            if other_trajs:
                consensus_ex_t = np.mean(other_trajs, axis=0)
                if np.std(traj) > 0 and np.std(consensus_ex_t) > 0:
                    correlations[t] = pearsonr(traj, consensus_ex_t)[0]
                else:
                    correlations[t] = 0
            else:
                correlations[t] = 1.0 
                
        top_tissues = sorted(correlations, key=correlations.get, reverse=True)[:3]
        
        for tissue, traj in tissue_trajs.items():
            if tissue not in top_tissues:
                y_smooth = smooth_trajectory(UNIQUE_MONTHS, traj, x_smooth, frac=0.5)
                ax.plot(x_smooth, y_smooth, color='#d3d3d3', alpha=0.5, linewidth=0.6, zorder=1)

        cons_smooth = smooth_trajectory(UNIQUE_MONTHS, consensus_traj, x_smooth, frac=0.6)
        std_smooth = smooth_trajectory(UNIQUE_MONTHS, consensus_std, x_smooth, frac=0.6)
        lower_bound = cons_smooth - std_smooth
        upper_bound = cons_smooth + std_smooth
        ax.fill_between(x_smooth, lower_bound, upper_bound, color='black', alpha=0.1, zorder=2, linewidth=0)

        end_positions = []
        for tissue in top_tissues:
            traj = tissue_trajs[tissue]
            y_sm = smooth_trajectory(UNIQUE_MONTHS, traj, x_smooth, frac=0.5)
            end_positions.append({'tissue': tissue, 'y': y_sm[-1], 'y_sm': y_sm})

        end_positions.sort(key=lambda item: item['y'])
        for i in range(1, len(end_positions)):
            if end_positions[i]['y'] - end_positions[i-1]['y'] < 0.35:
                end_positions[i]['y'] = end_positions[i-1]['y'] + 0.35

        for item in end_positions:
            tissue = item['tissue']
            y_smooth = item['y_sm']
            text_y = item['y']
            color = TISSUE_COLORS.get(tissue, '#333333')
            
            ax.plot(x_smooth, y_smooth, color=color, alpha=0.95, linewidth=1.2, zorder=3, path_effects=pe_white_border)
            clean_tissue_name = tissue.replace('_', ' ')
            

            ax.text(30.8, text_y, clean_tissue_name, color=color, fontsize=5.5,
                    fontname='Arial', fontweight='bold', va='center', zorder=5)

        ax.plot(x_smooth, cons_smooth, color='black', alpha=0.8, linewidth=1.0, linestyle='--', dashes=(4, 2), zorder=4, path_effects=pe_white_border_mean)

        if go_text:
            num_lines = go_text.count('\n') + 1
            dynamic_pad = 8 + num_lines * 8
            ax.set_title(f"Module {idx+1} (n={len(mod_genes)})", fontsize=7, fontweight='bold', color=title_color, pad=dynamic_pad)
            ax.text(0.5, 1.02, go_text, transform=ax.transAxes, ha='center', va='bottom', 
                    fontsize=6.5, color='#555555', fontweight='normal', linespacing=1.2)
        else:
            ax.set_title(f"Module {idx+1} (n={len(mod_genes)})", fontsize=7, fontweight='bold', color=title_color, pad=4)
        

        ax.set_xticks([0, 10, 20, 30])
        ax.set_xticklabels(["0", "10", "20", "30"])
        
        ax.set_yticks([-2, -1, 0, 1, 2])
        ax.set_yticklabels(["-2", "-1", "0", "1", "2"])
        

        ax.set_xlim(0, 42)
        ax.set_ylim(-2.8, 2.8) 
        ax.axhline(0, color='black', linestyle='-', linewidth=0.4, alpha=0.3, zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)


    patch_handles = []
    tissues_to_legend = sorted(actual_tissues_found)
    
    for tissue in tissues_to_legend:
        color = TISSUE_COLORS.get(tissue, '#333333')
        clean_tissue_name = tissue.replace('_', ' ')
        patch = mpatches.Patch(color=color, label=clean_tissue_name)
        patch_handles.append(patch)
                
    if patch_handles:
        fig.legend(handles=patch_handles, loc='upper center', bbox_to_anchor=(0.5, 0.97),
                   ncol=10, frameon=False, columnspacing=1.0, handlelength=0.7, handleheight=0.7)

    fig.text(0.5, 0.01, "Age (months)", ha='center', va='center', fontsize=7, fontweight='bold')
    fig.text(0.01, 0.5, "Scaled expression (Z-score)", ha='center', va='center', rotation='vertical', fontsize=7, fontweight='bold')


    fig.subplots_adjust(top=0.82, bottom=0.09, left=0.08, right=0.90, hspace=0.65, wspace=0.5)
    
    out_pdf = os.path.join(OUTPUT_PLOTS_DIR, "1-Figure_Perfect_Symmetry_Axes.pdf")
    with PdfPages(out_pdf) as pdf:
        pdf.savefig(fig, transparent=True)
    plt.close(fig)
    
    logger.info(f"✅ 刻度错觉已被物理修复！X轴起始锁定为0。请查看: {out_pdf}")

if __name__ == "__main__":
    main() 
    